In [ ]:
import pandas as pd
import numpy as np

In [ ]:
PATH = "<CLEANED_DATA_DIR>/"
trip_master = pd.read_csv(PATH + "trip_master_join.csv")
trip_exec = pd.read_excel(PATH + "route_execution_scores.xlsx")
gps_health = pd.read_csv(PATH + "gps_health_summary_v2.csv")

## Mapping Trip Execution Score to Respective Trip ID in trip_master_join

In [ ]:
trip_master['trip_execution_score'] = trip_master['trip_id'].map(
    trip_exec.set_index('trip_id')['score']
)
trip_master['trip_tier'] = trip_master['trip_id'].map(
    trip_exec.set_index('trip_id')['tier']
)
# condition for completed trips
mask = trip_master['status'] == 'completed'

# override values
trip_master.loc[mask, 'trip_execution_score'] = 100
trip_master.loc[mask, 'trip_tier'] = 'Excellent (>94)'

trip_master = trip_master.merge(
    gps_health,
    on='trip_id',
    how='left'   # keeps all rows from trip_master
)
trip_master.head()

In [ ]:
import matplotlib.pyplot as plt

# Count trips per tier
tier_counts = trip_master['trip_tier'].value_counts().reindex([
    'Excellent (>94)',
    'Good (>77)',
    'Partial (>50)',
    'Poor (<50)'
])

plt.figure(figsize=(8, 5))

# Plot bars (no explicit colors)
bars = plt.bar(tier_counts.index, tier_counts.values, width=0.45)

# Add labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 50,
        f"{int(height):,}" if not pd.isna(height) else "0",
        ha='center',
        fontsize=11
    )

plt.title("Performance Tier Distribution", fontsize=14, fontweight='bold')
plt.ylabel("Number of Trips")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Calculating Final Trip Score = mean(overall_health_score, 'trip_execution_score')

In [ ]:
trip_master['final_trip_score(gps+trip_exec)'] = (trip_master['overall_health_score']+trip_master['trip_execution_score'])/2

In [ ]:
trip_master.head()

## Dropping Trips with no Trip Stop Info

In [ ]:
# Identify trips to drop
dropped_trips = trip_master.loc[
    trip_master['final_trip_score(gps+trip_exec)'].isna(), 
    'trip_id'
]

# Drop those rows by matching trip_id
trip_master = trip_master[~trip_master['trip_id'].isin(dropped_trips)]

print(f"{len(dropped_trips.tolist())} trips dropped")



## Saving into a CSV File

In [ ]:
trip_master.to_csv(PATH + 'trip_score.csv')